<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔀 LangChain 动态 Middleware
### `wrap_model_call`、`wrap_tool_call`、`dynamic_prompt`

**一句话定位**：node 风格 hook 之外的另外两大家族——包住整次调用的 wrap 风格，和动态生成配置的 config 风格。

</div>

### 📖 这个 notebook 讲什么

这是一份 LangChain 里**非 hook 风格** middleware 的参考教程——它们**包住**真正的 LLM 调用或工具执行，或者**动态生成**提示词。这和 node 风格的钩子（`@before_agent`、`@after_agent` 等）在机制上完全不同。

### 🧠 Middleware 三大家族

| 家族 | 装饰器 | 拦截什么 | 类比 |
|---|---|---|---|
| 🪝 node 风格 | `@before_agent`、`@after_agent` | 节点边界的 state | Express 的 `app.use()` —— 只碰一侧 |
| 🔀 **wrap 风格** | `@wrap_model_call`、`@wrap_tool_call` | 真正的调用本身 | Python 装饰器 —— 包住整次调用 |
| ⚙️ **config 风格** | `@dynamic_prompt` | 提示词生成 | 模板引擎 —— 现场生成 |

### 🔑 关键区别

- **node 风格** = "在某个节点跑之前/之后，改一下 state"
- **wrap 风格** = "包住整次调用——一个函数里既看到进去的 request，又看到出来的 response"
- **config 风格** = "根据运行时状态动态生成配置（提示词、设置等）"

---

## 🔀 第一部分：`wrap_model_call`

### 它做什么

拦截 **LLM 调用本身**。拿到一个 `ModelRequest`（包含消息、工具、模型配置）和一个 `handler` 回调，可以：

- 🔄 **换模型** —— 根据对话状态路由到不同 LLM
- 🔧 **改参数** —— 改 temperature、max_tokens 等
- 🛡️ **过滤工具** —— 限制 LLM 能看到哪些工具
- ⏱️ **加计时/日志** —— 测 LLM 调用耗时
- 🔁 **实现重试** —— 捕获错误后换模型重试

### 写法模板

```python
@wrap_model_call
def my_wrapper(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    # 1. 检查 request（messages、tools、runtime 等）
    # 2. 可选：修改 request = request.override(model=..., tools=...)
    # 3. 调用 handler 继续往下走：response = handler(request)
    # 4. 可选：检查/修改 response
    return response
```

### `ModelRequest` 能读到什么

| 字段 | 类型 | 说明 |
|---|---|---|
| `request.messages` | `list[BaseMessage]` | `state["messages"]` 的快捷方式 |
| `request.runtime` | `Runtime` | 访问 `.context`、`.state`、`.config` |
| `request.tools` | `list[Tool]` | 当前 LLM 可用的工具 |
| `request.model` | `BaseChatModel` | 当前配置的模型 |

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### 🔄 例 1A：动态选模型

对话变长时路由到更大的模型。

**为什么？** 短对话用便宜快的模型就够了。但上下文一旦变长，就需要上下文窗口更大、推理更强的模型来处理堆积的历史。

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def route_by_length(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Switch to a larger model when conversation exceeds 10 messages."""
    message_count = len(request.messages)

    if message_count > 10:
        request = request.override(model=large_model)
    else:
        request = request.override(model=standard_model)

    return handler(request)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[route_by_length],
)

In [ ]:
from langchain.messages import HumanMessage

# Short conversation → should use gpt-5-nano
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 2 + 2?")]}
)
print("Model used:", response["messages"][-1].response_metadata["model_name"])
print("Answer:", response["messages"][-1].content)

In [ ]:
from langchain.messages import AIMessage

# Long conversation (11 messages) → should switch to claude-sonnet-4-5
response = agent.invoke(
    {"messages": [
        HumanMessage(content="Q1"), AIMessage(content="A1"),
        HumanMessage(content="Q2"), AIMessage(content="A2"),
        HumanMessage(content="Q3"), AIMessage(content="A3"),
        HumanMessage(content="Q4"), AIMessage(content="A4"),
        HumanMessage(content="Q5"), AIMessage(content="A5"),
        HumanMessage(content="What model are you?"),
    ]}
)
print("Model used:", response["messages"][-1].response_metadata["model_name"])

### 🛡️ 例 1B：动态过滤工具

`wrap_model_call` 也能根据运行时上下文**限制 LLM 能看到哪些工具**。这对**基于角色的访问控制（RBAC）**很有用——内部用户拿全部工具，外部用户只拿安全的子集。

关键方法：`request.override(tools=filtered_list)`

In [ ]:
from langchain.tools import tool
from dataclasses import dataclass


@tool
def web_search(query: str) -> str:
    """Search the web for public information."""
    return f"Web results for: {query}"


@tool
def internal_db_query(sql: str) -> str:
    """Query the internal database. Restricted to internal users."""
    return f"DB results for: {sql}"


@dataclass
class UserRole:
    user_role: str = "external"  # "internal" or "external"

In [ ]:
@wrap_model_call
def filter_tools_by_role(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """External users can only use web_search. Internal users get all tools."""
    user_role = request.runtime.context.user_role

    if user_role != "internal":
        # Strip internal tools — external users only see web_search
        safe_tools = [t for t in request.tools if t.name == "web_search"]
        request = request.override(tools=safe_tools)

    return handler(request)


agent_rbac = create_agent(
    model="gpt-5-nano",
    tools=[web_search, internal_db_query],
    middleware=[filter_tools_by_role],
    context_schema=UserRole,
)

In [ ]:
# External user tries to query the database — agent won't have that tool
response = agent_rbac.invoke(
    {"messages": [HumanMessage(content="How many users are in the database?")]},
    context=UserRole(user_role="external"),
)
print("External user:", response["messages"][-1].content)

---

## 🔀 第二部分：`wrap_tool_call`

### 它做什么

和 `wrap_model_call` 一样的模式，但包的是**工具执行**而不是 LLM 调用。拿到一个 `ToolCallRequest` 和一个 `handler`。

### 用途

| 用途 | 怎么做 |
|---|---|
| 🔒 授权 | 根据用户角色拦截工具调用 |
| 📊 日志/指标 | 统计每个工具耗时 |
| 💾 缓存 | 重复调用直接返回缓存结果 |
| 🔁 重试逻辑 | 失败的工具调用做退避重试 |
| 🛡️ 输入校验 | 执行前清洗工具参数 |

### 写法模板

```python
from langchain.agents.middleware import wrap_tool_call, ToolCallRequest

@wrap_tool_call
def my_tool_wrapper(
    request: ToolCallRequest,
    handler: Callable
):
    # request.tool_name —— 正在调用哪个工具
    # request.tool_args —— 参数
    # request.runtime   —— 访问 context、state

    # 可选：修改、拦截或走缓存
    result = handler(request)
    # 可选：改写结果
    return result
```

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**：查了 LangChain 官方文档（docs.langchain.com、reference.langchain.com）和 GitHub issue #33441，确认现在工具这一侧**只有** `wrap_tool_call` 这一个拦截式 hook，并**没有**对称的 `before_tool`/`after_tool` observational hook（这和 model 侧的 `before_model`/`after_model` 不一样，是不对称的）。LangChain 团队的立场是只在能带来新能力时才加新 hook，不为了"换一种写法"而加，所以短期内大概率不会补上。想在工具调用前后做轻量检查/记录，目前就用 `wrap_tool_call`，在 `handler(request)` 前后加逻辑即可，和下面例子里的写法完全一致。

</div>

### 例子：记录工具调用日志

In [ ]:
import time
from langchain.agents.middleware import wrap_tool_call, ToolCallRequest


@wrap_tool_call
def log_tool_calls(request: ToolCallRequest, handler: Callable):
    """Log every tool call with timing."""
    print(f"  🔨 Calling tool: {request.tool_name}")
    print(f"     Args: {request.tool_args}")

    start = time.time()
    result = handler(request)
    elapsed = time.time() - start

    print(f"  ✅ Tool {request.tool_name} completed in {elapsed:.2f}s")
    return result


agent_logged = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    middleware=[log_tool_calls],
)

In [ ]:
response = agent_logged.invoke(
    {"messages": [HumanMessage(content="Search for LangChain middleware docs")]}
)
print("\nFinal answer:", response["messages"][-1].content)

---

## ⚙️ 第三部分：`dynamic_prompt`

### 它做什么

根据运行时状态——到目前为止的对话、用户上下文、当前时间等——**动态生成系统提示词**。不再是写死的 `system_prompt="..."` 字符串，而是写一个返回提示词的函数。

### 为什么不直接用静态提示词？

| 静态提示词 | 动态提示词 |
|---|---|
| 所有用户一样 | 按用户个性化 |
| 每一轮都一样 | 对话中途也能变 |
| 读不到 context | 能读 `runtime.context` |
| 读不到 state | 能读 `request.messages`、`request.runtime.state` |

### 真实场景例子

- 🌍 **语言路由** —— 用用户的偏好语言回复
- 👤 **按角色行为** —— 管理员和普通用户拿到不同指令
- 🕐 **感知时间** —— "现在是非工作时间，建议改用 FAQ"
- 📊 **感知状态** —— "用户已经给过 X，别再问一遍"

### 写法模板

```python
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def my_prompt(request: ModelRequest) -> str:
    # 读任何需要的东西：context、state、messages
    user_lang = request.runtime.context.user_language
    return f"You are a helpful assistant. Respond in {user_lang}."
```

> 💡 `@dynamic_prompt` **替代**了 `create_agent()` 里的 `system_prompt` 参数。别两个都设——动态的那个会赢。

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class LanguageContext:
    user_language: str = "English"


@dynamic_prompt
def language_aware_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user's preferred language."""
    lang = request.runtime.context.user_language
    base = "You are a helpful assistant."

    if lang != "English":
        return f"{base} You MUST respond only in {lang}."
    return base


agent_lang = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[language_aware_prompt],
    # NOTE: no system_prompt= here — dynamic_prompt replaces it
)

In [ ]:
# English context → normal English response
response = agent_lang.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="English"),
)
print("English:", response["messages"][-1].content)

In [ ]:
# Spanish context → response in Spanish
response = agent_lang.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish"),
)
print("Spanish:", response["messages"][-1].content)

In [ ]:
# Japanese context → response in Japanese
response = agent_lang.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Japanese"),
)
print("Japanese:", response["messages"][-1].content)

---

## 🗺️ Middleware 完整速查表

### 基于装饰器的 middleware

| 装饰器 | 家族 | 拦截什么 | 关键方法 | 用途 |
|---|---|---|---|---|
| `@before_agent` | node | 整次运行开始前（只跑一次） | 返回 `dict` | 加载文件、校验输入 |
| `@after_agent` | node | 整次运行结束后（只跑一次） | 返回 `dict` | 保存结果、最终校验 |
| `@before_model` | node | 每次 LLM 调用前 | 返回 `dict` | 裁剪/总结消息 |
| `@after_model` | node | 每次 LLM 调用后 | 返回 `dict` | 人工审核、校验响应 |
| `@wrap_model_call` | wrap | LLM 调用本身 | `request.override()` | 换模型、过滤工具 |
| `@wrap_tool_call` | wrap | 工具执行 | `handler(request)` | 日志、缓存、授权 |
| `@dynamic_prompt` | config | 提示词生成 | 返回 `str` | 按用户/按轮次定制提示词 |

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**：这张表和官方文档（docs.langchain.com）一致——`before_agent`/`after_agent` 只在整次 agent 运行的开头/结尾各跑一次，`before_model`/`after_model` 才是每轮 LLM 调用都跑。工具侧确实**没有** `before_tool`/`after_tool`，只有 `wrap_tool_call`，和上面 Part 2 的说明对得上。

</div>

### 内置 middleware 类

| 类 | 作用 |
|---|---|
| `SummarizationMiddleware` | 自动总结旧消息 |
| `PIIMiddleware` | 检测/脱敏个人身份信息 |
| `HumanInTheLoopMiddleware` | 执行工具前暂停等人审批 |
| `ModelCallLimitMiddleware` | 限制每个 session 的 LLM 调用次数 |
| `ModelFallbackMiddleware` | 出错时回退到另一个模型 |
| `ModelRetryMiddleware` | LLM 调用失败时带退避重试 |
| `ToolCallLimitMiddleware` | 限制每个 session 的工具调用次数 |
| `ToolRetryMiddleware` | 工具调用失败重试 |
| `ShellToolMiddleware` | 沙箱化执行 shell 命令 |
| `ContextEditingMiddleware` | 编辑/清理上下文里的工具用量 |
| `LLMToolSelectorMiddleware` | 用一个 LLM 挑选该暴露哪些工具 |
| `LLMToolEmulator` | 用 LLM 模拟工具行为 |
| `TodoListMiddleware` | 用待办列表追踪任务 |
| `FilesystemFileSearchMiddleware` | 在文件系统里搜索文件 |

---

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：

1. **`wrap_model_call`** = 包住 LLM 调用，同时看到 request 和 response。最适合模型路由、工具过滤、重试。
2. **`wrap_tool_call`** = 包住工具执行，同样的模式。最适合日志、缓存、授权。
3. **`dynamic_prompt`** = 运行时生成系统提示词。最适合个性化、语言路由、按角色定制行为。
4. **这三者都能读** `request.runtime.context`（只读的用户数据）和 `request.runtime.state`（可变的 agent state）。
5. **可以自由组合** —— node 风格、wrap 风格、config 风格可以同时放进同一个 `middleware=[]` 列表，它们天然能组合在一起。

</div>